In [ ]:
# Extract and structure data formats using .
import os
import json
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

BASE_DIR = Path("/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset")

TRAIN_IMG = BASE_DIR / "train" / "image"
TRAIN_ANNO = BASE_DIR / "train" / "annos"

VAL_IMG = BASE_DIR / "validation" / "image"
VAL_ANNO = BASE_DIR / "validation" / "annos"

CLASS_MAP = {1:0, 2:1, 7:2, 8:3, 9:4} 

# YOLO structured dataset output directory
YOLO_BASE = Path("/kaggle/working/yolo_dataset")
IMG_DIR = YOLO_BASE / "images"
LBL_DIR = YOLO_BASE / "labels"

def convert_to_yolo(img_dir, anno_dir, split_name, limit=None):
    out_img_dir = IMG_DIR / split_name
    out_lbl_dir = LBL_DIR / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    # Sort files
    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith('.jpg') or f.lower().endswith('.png')])
    if limit:
        img_files = img_files[:limit]
        
    print(f"Structuring {len(img_files)} files for {split_name} split...")
    
    for img_name in tqdm(img_files):
        img_path = img_dir / img_name
        anno_name = img_name.rsplit('.', 1)[0] + '.json'
        anno_path = anno_dir / anno_name
        
        # Symlink image to YOLO directories to save runtime disk space
        dest_img_path = out_img_dir / img_name
        if not dest_img_path.exists():
            os.symlink(img_path, dest_img_path)
        
        # Read JSON and parse polygons into YOLO segmentation format (.txt)
        label_dest = out_lbl_dir / (img_name.rsplit('.', 1)[0] + '.txt')
        if label_dest.exists():
            continue
            
        yolo_lines = []
        if anno_path.exists():
            try:
                with open(anno_path, 'r') as f:
                    data = json.load(f)
                
                with Image.open(img_path) as im:
                    w, h = im.size
                
                for item in data.values():
                    cid = item.get("category_id")
                    if cid in CLASS_MAP:
                        yolo_cls = CLASS_MAP[cid]
                        segs = item.get("segmentation")
                        if not segs or not isinstance(segs[0], list): 
                            continue
                        
                        seg = segs[0]
                        # Need at least 3 pairs (6 values) for a valid bounding polygon
                        if len(seg) < 6: 
                            continue
                        
                        # Normalize polygon coordinates (clamping values [0, 1] to prevent loading errors)
                        norm_seg = []
                        for i in range(0, len(seg), 2):
                            nx = max(0.0, min(1.0, seg[i] / w))
                            ny = max(0.0, min(1.0, seg[i+1] / h))
                            norm_seg.extend([nx, ny])
                        
                        line = f"{yolo_cls} " + " ".join([f"{coord:.6f}" for coord in norm_seg])
                        yolo_lines.append(line)
                        
            except Exception as e:
                pass
        
        # Write format lines
        with open(label_dest, 'w') as f:
            f.write("\n".join(yolo_lines))

# Launch JSON to TXT Conversion (First 30000 / 5000 images constraint)
convert_to_yolo(TRAIN_IMG, TRAIN_ANNO, 'train', limit=30000)
convert_to_yolo(VAL_IMG, VAL_ANNO, 'val', limit=5000)

# Create custom dataset YAML
yaml_content = f"""path: {YOLO_BASE}
train: images/train
val: images/val
test: images/val

nc: 5
names: ['1', '2', '7', '8', '9']
"""
with open("/kaggle/working/custom_dataset.yaml", "w") as f:
    f.write(yaml_content)

print(f"Done! Dataset prepared in {YOLO_BASE}")

In [ ]:
!pip install ultralytics

In [ ]:
import os
import csv
import json
import math
import shutil
from pathlib import Path

import cv2
import yaml
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from ultralytics import YOLO

In [ ]:
from ultralytics import settings

print("Original Settings:", settings)
settings.update({"datasets_dir": "/kaggle/working/"})
print("Updated Settings:", settings)


In [ ]:
# =========================================================
# CONFIG
# =========================================================
DATA_YAML = Path("/kaggle/working/custom_dataset.yaml")

SAVE_DIR = Path("/kaggle/working/yolo_checkpoints")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

RUNS_DIR = Path("/kaggle/working/yolo_runs")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_METRICS_CSV = Path("/kaggle/working/training_metrics.csv")

MODEL_NAME = "yolov8n-seg.pt"    # transfer learning
RUN_NAME = "yolov8n_seg_exp"

EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 640
DEVICE = 0
PATIENCE = 20
SAVE_PERIOD = -1                
RESUME = False                   # set True to resume from last.pt manually if needed

# segmentation threshold for custom mask metrics
MASK_THRESHOLD = 0.5
PRED_CONF = 0.25
PRED_IOU = 0.5

In [ ]:
# =========================================================
# HELPERS
# =========================================================
def load_yaml(yaml_path):
    with open(yaml_path, "r") as f:
        return yaml.safe_load(f)


def get_names_from_yaml(yaml_path):
    data = load_yaml(yaml_path)
    names = data["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys())]
    return names


def ensure_metrics_csv(csv_path, class_names):
    columns = [
        "epoch",
        "train_box_loss",
        "train_seg_loss",
        "train_cls_loss",
        "train_dfl_loss",
        "val_box_p",
        "val_box_r",
        "val_box_map50",
        "val_box_map50_95",
        "val_seg_p",
        "val_seg_r",
        "val_seg_map50",
        "val_seg_map50_95",
        "val_miou_macro",
        "val_dice_macro",
    ]

    for cls_name in class_names:
        columns.append(f"iou_{cls_name}")
    for cls_name in class_names:
        columns.append(f"dice_{cls_name}")

    if not csv_path.exists():
        pd.DataFrame(columns=columns).to_csv(csv_path, index=False)


def append_metrics_row(csv_path, row_dict):
    df = pd.read_csv(csv_path)
    df = pd.concat([df, pd.DataFrame([row_dict])], ignore_index=True)
    df.to_csv(csv_path, index=False)


def polygon_to_mask(polygons, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)

    if polygons is None:
        return mask

    for poly in polygons:
        if poly is None or len(poly) < 6 or len(poly) % 2 != 0:
            continue
        pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
        cv2.fillPoly(mask, [pts], 1)

    return mask


def read_label_file_as_masks(label_path, img_h, img_w, num_classes):
    """
    Reads YOLO segmentation txt file and returns per-class GT masks.
    Combines all instances of the same class into one semantic mask per class.
    """
    gt_masks = [np.zeros((img_h, img_w), dtype=np.uint8) for _ in range(num_classes)]
    
    if not isinstance(label_path, Path):
        label_path = Path(label_path)

    if not label_path.exists():
        return gt_masks

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        parts = line.split()
        if len(parts) < 7:
            continue

        cls_id = int(float(parts[0]))
        coords = list(map(float, parts[1:]))

        if len(coords) % 2 != 0:
            continue

        poly = []
        for i in range(0, len(coords), 2):
            x = min(max(coords[i] * img_w, 0), img_w - 1)
            y = min(max(coords[i + 1] * img_h, 0), img_h - 1)
            poly.extend([x, y])

        instance_mask = polygon_to_mask([poly], img_h, img_w)
        gt_masks[cls_id] = np.maximum(gt_masks[cls_id], instance_mask)

    return gt_masks


def build_pred_class_masks(result, img_h, img_w, num_classes, threshold=0.5):
    """
    Builds per-class predicted semantic masks by unioning all instance masks of same class.
    """
    pred_masks = [np.zeros((img_h, img_w), dtype=np.uint8) for _ in range(num_classes)]

    if result.masks is None or result.boxes is None:
        return pred_masks

    masks = result.masks.data.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)

    for i in range(len(classes)):
        cls_id = classes[i]
        mask = (masks[i] > threshold).astype(np.uint8)

        if mask.shape != (img_h, img_w):
            mask = cv2.resize(mask, (img_w, img_h), interpolation=cv2.INTER_NEAREST)
            mask = (mask > 0).astype(np.uint8)

        pred_masks[cls_id] = np.maximum(pred_masks[cls_id], mask)

    return pred_masks


def iou_score(gt, pred):
    intersection = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()
    if union == 0:
        return 1.0
    return intersection / union


def dice_score(gt, pred):
    intersection = np.logical_and(gt, pred).sum()
    denom = gt.sum() + pred.sum()
    if denom == 0:
        return 1.0
    return (2.0 * intersection) / denom


def compute_segmentation_metrics(model, data_yaml, split="val", conf=0.25, iou=0.5, threshold=0.5):
    data = load_yaml(data_yaml)
    num_classes = data["nc"]
    class_names = get_names_from_yaml(data_yaml)

    # Get the split path from custom_yaml mapping 
    # e.g., 'images/val' based off the YOLO root
    split_path = Path(data["path"]) / data[split]

    image_paths = []
    # If the configured split is a directory
    if split_path.is_dir():
        image_paths = sorted(list(split_path.glob("*.jpg")) + list(split_path.glob("*.png")) + list(split_path.glob("*.jpeg")))
    # If it's a direct index txt list
    elif split_path.is_file():
        with open(split_path, "r") as f:
            image_paths = [Path(line.strip()) for line in f.read().splitlines() if line.strip()]

    iou_scores_per_class = [[] for _ in range(num_classes)]
    dice_scores_per_class = [[] for _ in range(num_classes)]

    for img_path in tqdm(image_paths, desc=f"Computing {split} mIoU/Dice"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_h, img_w = img.shape[:2]

        # Because our directory is set up with sibling 'images' / 'labels' folds
        label_path = Path(str(img_path).replace('images', 'labels')).with_suffix(".txt")

        gt_class_masks = read_label_file_as_masks(label_path, img_h, img_w, num_classes)

        results = model.predict(
            source=str(img_path),
            imgsz=IMG_SIZE,
            conf=conf,
            iou=iou,
            verbose=False,
            retina_masks=True,
            device=DEVICE
        )

        if len(results) == 0:
            pred_class_masks = [np.zeros((img_h, img_w), dtype=np.uint8) for _ in range(num_classes)]
        else:
            pred_class_masks = build_pred_class_masks(results[0], img_h, img_w, num_classes, threshold=threshold)

        for cls_id in range(num_classes):
            gt = gt_class_masks[cls_id].astype(bool)
            pred = pred_class_masks[cls_id].astype(bool)

            iou_val = iou_score(gt, pred)
            dice_val = dice_score(gt, pred)

            iou_scores_per_class[cls_id].append(iou_val)
            dice_scores_per_class[cls_id].append(dice_val)

    iou_per_class = {
        class_names[i]: float(np.mean(iou_scores_per_class[i])) if len(iou_scores_per_class[i]) else 0.0
        for i in range(num_classes)
    }

    dice_per_class = {
        class_names[i]: float(np.mean(dice_scores_per_class[i])) if len(dice_scores_per_class[i]) else 0.0
        for i in range(num_classes)
    }

    miou_macro = float(np.mean(list(iou_per_class.values()))) if len(iou_per_class) else 0.0
    dice_macro = float(np.mean(list(dice_per_class.values()))) if len(dice_per_class) else 0.0

    return iou_per_class, dice_per_class, miou_macro, dice_macro


def find_latest_weights(run_dir):
    weights_dir = run_dir / "weights"
    best_pt = weights_dir / "best.pt"
    last_pt = weights_dir / "last.pt"
    return best_pt, last_pt


def read_results_csv(run_dir):
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        return None
    return pd.read_csv(results_csv)

In [ ]:
# =========================================================
# TRAIN
# =========================================================
class_names = get_names_from_yaml(DATA_YAML)
ensure_metrics_csv(TRAINING_METRICS_CSV, class_names)

model = YOLO(MODEL_NAME)

def on_fit_epoch_end(trainer):
   
    run_dir = Path(trainer.save_dir)
    weights_dir = run_dir / 'weights'
    
    best_pt = weights_dir / 'best.pt'
    last_pt = weights_dir / 'last.pt'
    
    if best_pt.exists():
        shutil.copy2(best_pt, SAVE_DIR / 'best_model.pt')
    if last_pt.exists():
        shutil.copy2(last_pt, SAVE_DIR / 'yolo_checkpoint_last.pt')
        
    results_csv = run_dir / 'results.csv'
    metrics_safe = Path("/kaggle/working/yolo_metrics_backup.csv")
    if results_csv.exists():
        shutil.copy2(results_csv, metrics_safe)

model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

try:
    train_result = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        patience=PATIENCE,
        save=True,
        save_period=SAVE_PERIOD,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        pretrained=True,
        optimizer="AdamW",
        lr0=1e-3,
        lrf=1e-2,
        weight_decay=5e-4,
        warmup_epochs=3,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=5.0,
        translate=0.1,
        scale=0.2,
        fliplr=0.5,
        mosaic=0.5,
        mixup=0.0,
        copy_paste=0.0,
        resume=RESUME,
        plots=True,
    )
except KeyboardInterrupt:
    print("\n[INFO] Training stopped manually via interrupt.")
except BaseException as e:
    print(f"\n[ERROR] Training interrupted due to error/signal: {e}")
finally:
    print("\n[INFO] Finalizing Checkpoint Backups...")
    run_dir = RUNS_DIR / RUN_NAME
    best_pt, last_pt = find_latest_weights(run_dir)

    if best_pt.exists():
        shutil.copy2(best_pt, SAVE_DIR / "best_model.pt")
        print(f"Backed up best_model.pt to {SAVE_DIR}")
    if last_pt.exists():
        shutil.copy2(last_pt, SAVE_DIR / "yolo_checkpoint_last.pt")
        print(f"Backed up yolo_checkpoint_last.pt to {SAVE_DIR}")

    print("\nTraining workflow complete. Your models are safely in:", SAVE_DIR)


## Notes